In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import os

In [9]:



folder_path = r"individual_stock_data"
output_folder = r"Var/CVaR_Results"
CLOSE_COL   = "Close"
DATE_COL    = "Date"
CONFIDENCE  = 0.95
PORTFOLIO   = 1_000_000
EXPORT_CSV  = True
def load_returns(filepath):
    try:       
        df = pd.read_csv(filepath, parse_dates=[DATE_COL])
        df      = df.sort_values(DATE_COL).reset_index(drop=True)
        prices  = pd.to_numeric(df[CLOSE_COL], errors="coerce").dropna()# Convert to numeric and drop non-numeric values
        returns = np.log(prices / prices.shift(1)).dropna()#Calculate long returns by calculating the price on a cerian day divided by the price on the previous day and then taking the log of that value
        return returns
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None
#Var
def historical_var(returns, alpha):
    return float(-np.percentile(returns, (1 - alpha) * 100))
 
def parametric_var(returns, alpha):
    z = stats.norm.ppf(alpha)
    return float(z * returns.std() - returns.mean())
 
def cornish_fisher_var(returns, alpha):
    mu, sigma, S, K = returns.mean(), returns.std(), returns.skew(), returns.kurt()
    z    = stats.norm.ppf(alpha)
    z_cf = (z
            + (z**2 - 1) * S / 6
            + (z**3 - 3*z) * K / 24
            - (2*z**3 - 5*z) * S**2 / 36)
    return float(z_cf * sigma - mu)
#CVaR
def historical_cvar(returns, alpha):
    tail = returns[returns < -historical_var(returns, alpha)]
    return float(-tail.mean())
 
def parametric_cvar(returns, alpha):
    z = stats.norm.ppf(alpha)
    return float(returns.std() * stats.norm.pdf(z) / (1 - alpha) - returns.mean())
 
def cornish_fisher_cvar(returns, alpha):
    mu, sigma, S, K = returns.mean(), returns.std(), returns.skew(), returns.kurt()
    z    = stats.norm.ppf(alpha)
    z_cf = (z
            + (z**2 - 1) * S / 6
            + (z**3 - 3*z) * K / 24
            - (2*z**3 - 5*z) * S**2 / 36)
    return float(sigma * stats.norm.pdf(z_cf) / (1 - alpha) - mu)
def process(ticker, returns, alpha, portfolio):
    h_var   = historical_var(returns, alpha)
    p_var   = parametric_var(returns, alpha)
    cf_var  = cornish_fisher_var(returns, alpha)
    h_cvar  = historical_cvar(returns, alpha)
    p_cvar  = parametric_cvar(returns, alpha)
    cf_cvar = cornish_fisher_cvar(returns, alpha)
 
    print(f"  {'Method':<20} {'VaR':>8}  {'VaR (Rs)':>12}  {'CVaR':>8}  {'CVaR (Rs)':>12}")
    print(f"  {'─'*66}")
    for label, var, cvar in [
        ("Historical",     h_var,  h_cvar),
        ("Parametric",     p_var,  p_cvar),
        ("Cornish-Fisher", cf_var, cf_cvar),
    ]:
        print(f"  {label:<20} {var*100:>7.3f}%  Rs{var*portfolio:>10,.0f}  "
              f"{cvar*100:>7.3f}%  Rs{cvar*portfolio:>10,.0f}")
 
    return {
        "Ticker":        ticker,
        "N_days":        len(returns),
        "Mean_%":        round(returns.mean() * 100, 4),
        "Std_%":         round(returns.std()  * 100, 4),
        "Skewness":      round(returns.skew(), 4),
        "Kurt_excess":   round(returns.kurt(), 4),
        "Hist_VaR_%":    round(h_var   * 100, 4),
        "Param_VaR_%":   round(p_var   * 100, 4),
        "CF_VaR_%":      round(cf_var  * 100, 4),
        "Hist_CVaR_%":   round(h_cvar  * 100, 4),
        "Param_CVaR_%":  round(p_cvar  * 100, 4),
        "CF_CVaR_%":     round(cf_cvar * 100, 4),
        "Hist_VaR_Rs":   round(h_var   * portfolio, 2),
        "Param_VaR_Rs":  round(p_var   * portfolio, 2),
        "CF_VaR_Rs":     round(cf_var  * portfolio, 2),
        "Hist_CVaR_Rs":  round(h_cvar  * portfolio, 2),
        "Param_CVaR_Rs": round(p_cvar  * portfolio, 2),
        "CF_CVaR_Rs":    round(cf_cvar * portfolio, 2),
    }
def main():
    csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])
 
    if not csv_files:
        print(f"No CSV files found in '{folder_path}'. Check the folder path.")
        return
 
    print(f"\nFound {len(csv_files)} CSV files in '{folder_path}'.")
    print(f"Confidence: {CONFIDENCE*100:.0f}%  |  Portfolio: Rs{PORTFOLIO:,.0f}")
    print("=" * 68)
 
    results, failed = [], []
 
    for i, filename in enumerate(csv_files, 1):
        ticker   = os.path.splitext(filename)[0]
        filepath = os.path.join(folder_path, filename)
 
        print(f"\n[{i}/{len(csv_files)}] {ticker}")
        returns = load_returns(filepath)
 
        if returns is None:
            failed.append(ticker)
            continue
 
        print(f"  Days: {len(returns)}  |  "
              f"Skew: {returns.skew():.3f}  |  Kurt: {returns.kurt():.3f}")
        results.append(process(ticker, returns, CONFIDENCE, PORTFOLIO))
    if results:
    
        summary = pd.DataFrame(results)

        print(f"\nDone — {len(results)} stocks processed, {len(failed)} skipped")
        if failed:
            print(f"Skipped: {', '.join(failed)}")

        if EXPORT_CSV:
            out = os.path.join(output_folder, "var_cvar_results.csv")
            summary.to_csv(out, index=False)
            print(f"Results saved to: {out}") 
if __name__ == "__main__":
    main()       



Found 200 CSV files in 'individual_stock_data'.
Confidence: 95%  |  Portfolio: Rs1,000,000

[1/200] 63MOONS
  Days: 1247  |  Skew: 0.692  |  Kurt: 1.648
  Method                    VaR      VaR (Rs)      CVaR     CVaR (Rs)
  ──────────────────────────────────────────────────────────────────
  Historical             5.120%  Rs    51,200    5.440%  Rs    54,400
  Parametric             5.397%  Rs    53,967    6.811%  Rs    68,113
  Cornish-Fisher         5.920%  Rs    59,196    5.180%  Rs    51,804

[2/200] AARTIDRUGS
  Days: 1247  |  Skew: 1.553  |  Kurt: 9.890
  Method                    VaR      VaR (Rs)      CVaR     CVaR (Rs)
  ──────────────────────────────────────────────────────────────────
  Historical             2.978%  Rs    29,784    4.359%  Rs    43,592
  Parametric             3.760%  Rs    37,600    4.702%  Rs    47,025
  Cornish-Fisher         4.204%  Rs    42,035    3.352%  Rs    33,523

[3/200] ADVENZYMES
  Days: 1247  |  Skew: 0.594  |  Kurt: 6.716
  Method          